# [Elasticsearch VectorDB](https://www.elastic.co/elasticsearch/)


## 1. Elasticsearch란?
> Elasticsearch는 Apache Lucene 기반의 분산형 RESTful 검색 및 분석 엔진으로, 벡터 검색과 전통적인 키워드 검색을 모두 지원하는 강력한 데이터베이스입니다.


### 주요 특징
- **하이브리드 검색**: 벡터 검색(Semantic)과 키워드 검색(Lexical)을 동시에 지원
- **실시간 검색**: 준실시간(Near Real-Time) 검색 및 분석 기능
- **확장성**: 수평적 확장이 용이한 분산 아키텍처
- **RESTful API**: 직관적인 HTTP REST API 제공
- **강력한 분석 기능**: Kibana를 통한 시각화 및 대시보드
- **다양한 검색 전략**: Dense Vector, Sparse Vector(ELSER), BM25 등 지원
- **LangChain 통합**: LangChain과 완벽하게 통합되어 RAG 시스템 구축 가능


### 사용 사례
- 엔터프라이즈 검색 시스템
- 로그 및 이벤트 데이터 분석
- RAG (Retrieval Augmented Generation) 시스템
- 하이브리드 검색 (의미 검색 + 키워드 검색)
- 실시간 애플리케이션 성능 모니터링
- 전자상거래 상품 검색


## 2. Elasticsearch vs 다른 벡터 데이터베이스


### Elasticsearch의 장점
- **하이브리드 검색**: 벡터 검색과 전통적인 텍스트 검색을 동시에 활용 가능
- **성숙한 생태계**: Elastic Stack (Elasticsearch, Logstash, Kibana, Beats)의 풍부한 도구
- **강력한 분석**: 집계(Aggregation) 기능을 통한 복잡한 데이터 분석
- **검증된 안정성**: 수년간 운영된 엔터프라이즈급 안정성
- **다양한 배포 옵션**: Self-hosted, Elastic Cloud 등
- **실시간 업데이트**: 준실시간 인덱싱 및 검색


### 다른 솔루션과의 비교

| 특성 | Chroma | pgvector | Qdrant | Milvus | Elasticsearch |
|------|--------|----------|--------|--------|---------------|
| **복잡도** | 낮음 | 중간 | 중간 | 높음 | 중간 |
| **설정 난이도** | 매우 쉬움 | 쉬움 | 쉬움 | 어려움 | 중간 |
| **확장성** | 소규모 | 중규모 | 중대규모 | 대규모 | 대규모 |
| **검색 속도** | 보통 | 보통 | 빠름 | 매우 빠름 | 빠름 |
| **하이브리드 검색** | 제한적 | 가능 | 가능 | 가능 | **탁월함** |
| **분석 기능** | 없음 | 기본 | 제한적 | 제한적 | **매우 강력** |
| **생태계** | 신생 | PostgreSQL | 성장 중 | 성장 중 | **매우 성숙** |
| **운영 경험** | 부족 | 풍부 | 중간 | 중간 | **매우 풍부** |
| **메모리 효율** | 보통 | 좋음 | 좋음 | 매우 좋음 | 좋음 |
| **기존 데이터 통합** | 어려움 | **쉬움** | 중간 | 중간 | **쉬움** |



#### 선택 가이드
- **Chroma**: 빠른 프로토타이핑, 소규모 프로젝트
- **pgvector**: 기존 PostgreSQL 인프라 활용, 중간 규모
- **Qdrant**: 고성능 벡터 검색, 풍부한 필터링
- **Milvus**: 초대규모 벡터 데이터, 최고 성능 요구
- **Elasticsearch**: 하이브리드 검색, 복잡한 분석, 엔터프라이즈 환경, 기존 Elastic Stack 활용


## 3. 설치 및 설정

Elasticsearch를 Docker Compose를 사용하여 설치하겠습니다.


### 3.1 Docker Compose로 Elasticsearch 실행

`elasticsearch/docker-compose.yml` 파일을 사용하여 Elasticsearch와 Kibana를 실행합니다.

```bash
cd elasticsearch
docker-compose up -d
```

서비스가 정상적으로 실행되면:
- Elasticsearch REST API: http://localhost:9200
- Kibana Web UI: http://localhost:5601


### 3.2 Elasticsearch 연결 확인


In [ ]:
import requests

# Elasticsearch 클러스터 상태 확인
response = requests.get("http://localhost:9200/_cluster/health")
print("클러스터 상태:", response.json())

# Elasticsearch 버전 정보
response = requests.get("http://localhost:9200")
print("\nElasticsearch 정보:", response.json())


### 3.3 Kibana Web UI

브라우저에서 http://localhost:5601 접속하여 Kibana를 통해 Elasticsearch를 시각적으로 관리할 수 있습니다.

주요 기능:
- Dev Tools: Elasticsearch API를 직접 테스트
- Discover: 데이터 탐색 및 검색
- Dashboard: 데이터 시각화
- Index Management: 인덱스 관리


## 4. LangChain과 Elasticsearch 통합

실제 텍스트 파일을 로드하여 Elasticsearch에 저장하고 검색하는 완전한 예제를 만들어보겠습니다.


### 4.1 텍스트 파일 로드


In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("./data/rag-keywords.txt", encoding="utf-8")
documents = loader.load()
print(f"파일 로드 완료: {len(documents)}개 문서")


### 4.2 텍스트 분할


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

splits = text_splitter.split_documents(documents)
print(f"텍스트 분할 완료: {len(splits)}개 청크")


### 4.3 임베딩 모델 설정


In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b")


### 4.4 Elasticsearch 벡터스토어 생성

LangChain의 `ElasticsearchStore`를 사용하여 벡터스토어를 생성합니다.


In [ ]:
from langchain_elasticsearch import ElasticsearchStore

# Elasticsearch 벡터스토어 생성
vectorstore = ElasticsearchStore.from_documents(
    documents=splits,                    # 분할된 문서
    embedding=embeddings,                # 임베딩 함수
    es_url="http://localhost:9200",     # Elasticsearch URL
    index_name="rag_keywords",          # 인덱스 이름
)

print("Elasticsearch 벡터스토어 생성 완료")


### 4.5 인덱스 확인

생성된 인덱스를 확인합니다.


In [ ]:
# 인덱스 목록 확인
response = requests.get("http://localhost:9200/_cat/indices?v")
print(response.text)

# rag_keywords 인덱스 정보 확인
response = requests.get("http://localhost:9200/rag_keywords/_count")
print(f"\n저장된 문서 수: {response.json()['count']}")


## 5. 검색 기능

Elasticsearch의 다양한 검색 기능을 알아보겠습니다.


### 5.1 유사도 검색 (Similarity Search)


In [ ]:
# 기본 유사도 검색
query = "딥러닝이란 무엇인가요?"
results = vectorstore.similarity_search(query, k=3)

print(f"검색 쿼리: {query}\n")
print("검색 결과:")
print("-" * 50)
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content}")
    print(f"   메타데이터: {doc.metadata}")
    print()


### 5.2 유사도 점수와 함께 검색


In [ ]:
# 유사도 점수와 함께 검색
query = "인공지능의 정의"
results = vectorstore.similarity_search_with_score(query, k=3)

print(f"검색 쿼리: {query}\n")
print("검색 결과 (점수 포함):")
print("-" * 50)
for i, (doc, score) in enumerate(results, 1):
    print(f"{i}. [유사도: {score:.4f}] {doc.page_content[:100]}...")
    print()


### 5.3 샘플 문서를 이용한 고급 검색

메타데이터 필터링을 테스트하기 위해 샘플 문서를 추가합니다.


In [ ]:
from langchain_core.documents import Document

# 샘플 문서 생성
sample_documents = [
    Document(
        page_content="인공지능은 인간의 지능을 모방하는 기술입니다.",
        metadata={"category": "AI", "topic": "기본개념", "source": "ai_basics.txt"}
    ),
    Document(
        page_content="머신러닝은 데이터로부터 패턴을 학습하는 AI의 한 분야입니다.",
        metadata={"category": "ML", "topic": "학습방법", "source": "ml_intro.txt"}
    ),
    Document(
        page_content="딥러닝은 신경망을 사용하는 머신러닝의 하위 분야입니다.",
        metadata={"category": "DL", "topic": "신경망", "source": "dl_guide.txt"}
    ),
    Document(
        page_content="자연어처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.",
        metadata={"category": "NLP", "topic": "언어처리", "source": "nlp_handbook.txt"}
    )
]

# 샘플 문서를 위한 별도의 벡터스토어
sample_vectorstore = ElasticsearchStore.from_documents(
    documents=sample_documents,
    embedding=embeddings,
    es_url="http://localhost:9200",
    index_name="sample_docs",
)

print(f"샘플 문서 {len(sample_documents)}개가 저장되었습니다.")


### 5.4 메타데이터 필터링 검색


In [ ]:
# 특정 카테고리만 검색 (AI 카테고리)
query = "인공지능 기술"
results = sample_vectorstore.similarity_search(
    query, 
    k=3,
    filter=[{"term": {"metadata.category.keyword": "AI"}}]
)

print(f"검색 쿼리: {query}")
print("필터 조건: category = 'AI'")
print("-" * 50)
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content}")
    print(f"   카테고리: {doc.metadata['category']}")
    print()


## 6. Elasticsearch의 다양한 검색 전략

Elasticsearch는 여러 가지 검색 전략을 제공합니다.


### 6.1 Dense Vector Strategy (기본)

HNSW(Hierarchical Navigable Small World) 알고리즘을 사용한 근사 최근접 이웃 검색입니다.


In [ ]:
from langchain_elasticsearch import DenseVectorStrategy

# Dense Vector Strategy (기본값)
dense_vectorstore = ElasticsearchStore(
    es_url="http://localhost:9200",
    index_name="dense_vector_test",
    embedding=embeddings,
    strategy=DenseVectorStrategy()  # 기본 전략
)

print("Dense Vector Strategy 벡터스토어 생성 완료")


### 6.2 BM25 Strategy (키워드 검색)

전통적인 키워드 기반 검색 (임베딩 없이 사용 가능)


In [ ]:
from langchain_elasticsearch import BM25Strategy

# BM25 Strategy (키워드 검색)
bm25_vectorstore = ElasticsearchStore(
    es_url="http://localhost:9200",
    index_name="bm25_test",
    strategy=BM25Strategy()  # 임베딩 불필요
)

# 텍스트만 추가 (임베딩 없이)
bm25_vectorstore.add_texts(
    ["딥러닝은 인공신경망을 사용합니다", 
     "머신러닝은 데이터로부터 학습합니다",
     "자연어처리는 언어를 이해하는 기술입니다"]
)

# BM25 검색
results = bm25_vectorstore.similarity_search("신경망", k=2)
print("BM25 검색 결과:")
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content}")


### 6.3 Hybrid Search (벡터 + BM25)

Elasticsearch의 강력한 기능 중 하나인 하이브리드 검색입니다.


In [ ]:
# Hybrid Strategy (벡터 + BM25)
hybrid_vectorstore = ElasticsearchStore.from_documents(
    documents=sample_documents,
    embedding=embeddings,
    es_url="http://localhost:9200",
    index_name="hybrid_search",
    strategy=DenseVectorStrategy(hybrid=True)  # 하이브리드 모드
)

# 하이브리드 검색
query = "신경망 딥러닝"
results = hybrid_vectorstore.similarity_search(query, k=2)

print(f"하이브리드 검색 결과: '{query}'")
print("-" * 50)
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content}")
    print()


## 7. LCEL 방식의 RAG 체인

Elasticsearch와 LLM을 결합하여 실제 질의응답 시스템을 구축해보겠습니다.


### 7.1 LLM 설정


In [ ]:
from langchain_ollama.chat_models import ChatOllama

llm = ChatOllama(
    model="gemma3:4b",
    temperature=0.1,
    top_p=1.0,
    num_predict=256,
    keep_alive="5m"
)


### 7.2 RAG 체인 구성


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Retriever 설정
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_template("""
    Answer the question based on the context:

    <context>
    {context}
    </context>

    Question: {question}
""")

# 문서를 문자열로 변환하는 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# LCEL 방식의 RAG 체인 구성
qa_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG 체인 구성 완료")


### 7.3 질문 답변 실행


In [ ]:
# 질문 답변 실행
question = "딥러닝이란 무엇인가요?"
answer = qa_chain.invoke(question)

print(f"질문: {question}")
print(f"답변: {answer}")


### 7.4 다양한 질문 테스트


In [ ]:
def ask_question(qa_chain, question):
    """질문에 대한 답변을 출력하는 함수"""
    print(f"\n{'='*60}")
    print(f"질문: {question}")
    print(f"{'='*60}")
    
    answer = qa_chain.invoke(question)
    print(f"답변: {answer}")
    print(f"{'='*60}\n")
    
    return answer

# 여러 질문 테스트
questions = [
    "벡터 데이터베이스란 무엇인가요?",
    "임베딩의 역할은 무엇인가요?",
    "RAG 시스템은 어떻게 작동하나요?"
]

for q in questions:
    ask_question(qa_chain, q)


## 8. 고급 기능

Elasticsearch의 고급 기능들을 알아보겠습니다.


### 8.1 커스텀 쿼리 (Custom Query)

Elasticsearch의 강력한 쿼리 DSL을 직접 사용할 수 있습니다.


In [ ]:
def custom_query(query_body: dict, query: str) -> dict:
    """커스텀 Elasticsearch 쿼리"""
    print("기본 쿼리:")
    print(query_body)
    print()
    
    # BM25 부스팅을 추가한 커스텀 쿼리
    custom_query_body = {
        "query": {
            "bool": {
                "should": [
                    query_body["query"],  # 원래 벡터 검색
                    {
                        "match": {
                            "text": {
                                "query": query,
                                "boost": 0.5  # BM25 가중치
                            }
                        }
                    }
                ]
            }
        }
    }
    
    print("커스텀 쿼리:")
    print(custom_query_body)
    print()
    
    return custom_query_body

# 커스텀 쿼리 사용
# results = vectorstore.similarity_search(
#     "딥러닝 신경망",
#     k=3,
#     custom_query=custom_query
# )


### 8.2 인덱스 설정 최적화

대용량 데이터 처리를 위한 인덱스 설정입니다.


In [ ]:
# 벌크 추가 시 청크 크기 조정
# 타임아웃 에러 발생 시 유용

large_documents = splits * 10  # 예시: 많은 문서

vectorstore.add_texts(
    [doc.page_content for doc in large_documents],
    bulk_kwargs={
        "chunk_size": 50,        # 기본값: 500
        "max_chunk_bytes": 200000000  # 기본값: 100MB
    }
)

print("대용량 문서 추가 완료")


### 8.3 인덱스 통계 및 모니터링


In [ ]:
import json

# 인덱스 통계
response = requests.get("http://localhost:9200/rag_keywords/_stats")
stats = response.json()

print("인덱스 통계:")
print(f"문서 수: {stats['_all']['primaries']['docs']['count']}")
print(f"삭제된 문서: {stats['_all']['primaries']['docs']['deleted']}")
print(f"저장 크기: {stats['_all']['primaries']['store']['size_in_bytes'] / 1024 / 1024:.2f} MB")

# 매핑 정보
response = requests.get("http://localhost:9200/rag_keywords/_mapping")
print("\n인덱스 매핑:")
print(json.dumps(response.json(), indent=2))


### 8.4 인덱스 삭제 및 관리


In [ ]:
# 테스트 인덱스 삭제
test_indices = ["dense_vector_test", "bm25_test", "hybrid_search"]

for index in test_indices:
    response = requests.delete(f"http://localhost:9200/{index}")
    if response.status_code == 200:
        print(f"인덱스 '{index}' 삭제 완료")
    elif response.status_code == 404:
        print(f"인덱스 '{index}' 존재하지 않음")
    else:
        print(f"인덱스 '{index}' 삭제 실패: {response.text}")


## 9. Elasticsearch의 장단점 정리


### 장점

1. **하이브리드 검색의 강자**
   - 벡터 검색과 키워드 검색을 동시에 사용 가능
   - BM25와 Dense Vector를 결합한 최적의 검색 결과

2. **성숙한 생태계**
   - Kibana를 통한 강력한 시각화
   - Logstash, Beats 등 데이터 수집 도구
   - 풍부한 플러그인 및 커뮤니티

3. **엔터프라이즈급 안정성**
   - 수년간 검증된 안정성
   - 분산 아키텍처로 고가용성 보장
   - 자동 샤딩 및 레플리케이션

4. **강력한 분석 기능**
   - 집계(Aggregation) 기능
   - 실시간 분석 및 대시보드
   - 복잡한 쿼리 DSL

5. **유연한 배포 옵션**
   - Self-hosted, Elastic Cloud
   - 다양한 클라우드 플랫폼 지원

### 단점

1. **메모리 사용량**
   - JVM 기반으로 메모리 사용량이 높음
   - 적절한 힙 크기 설정 필요

2. **복잡한 설정**
   - 최적의 성능을 위해서는 많은 튜닝 필요
   - 초보자에게는 진입 장벽이 있음

3. **리소스 요구사항**
   - 소규모 프로젝트에는 과할 수 있음
   - 최소 2GB 이상의 메모리 권장

4. **라이선스 변경**
   - 최근 Apache 2.0에서 SSPL/Elastic License로 변경
   - 일부 사용 사례에서 제약 가능

### 추천 사용 사례

- ✅ 엔터프라이즈급 검색 시스템
- ✅ 하이브리드 검색이 필요한 경우
- ✅ 기존 Elastic Stack 인프라 활용
- ✅ 복잡한 분석 및 시각화 필요
- ✅ 로그 분석 + 벡터 검색 결합
- ❌ 간단한 프로토타입 (→ Chroma 추천)
- ❌ 초대규모 벡터 전용 (→ Milvus 추천)
- ❌ 리소스가 제한된 환경


## 10. 참고 자료

- [Elasticsearch 공식 문서](https://www.elastic.co/guide/en/elasticsearch/reference/current/index.html)
- [LangChain Elasticsearch 통합](https://docs.langchain.com/oss/python/integrations/vectorstores/elasticsearch)
- [Elasticsearch Vector Search](https://www.elastic.co/guide/en/elasticsearch/reference/current/knn-search.html)
- [Kibana 사용 가이드](https://www.elastic.co/guide/en/kibana/current/index.html)
